# 4.1 Code Brief: Systematic Model ComparisonThis notebook contains a condensed reference of the key code patterns from notebook 4.1. Use it as a quick reference.

## Key Pattern: Instantiate → Fit → Predict```python# Same pattern for loading and using any tuned model:model = joblib.load(model_path)y_prob = model.predict_proba(X_test)[:, 1]```

## Setup and Data Preparation

In [ ]:
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'data_filepath = '/data/'course3_models = '/models/'import numpy as npimport pandas as pdimport joblibimport warningswarnings.filterwarnings('ignore')import plotly.graph_objects as gofrom sklearn.metrics import (precision_score, recall_score, f1_score,    precision_recall_curve, average_precision_score, brier_score_loss)RANDOM_STATE = 42np.random.seed(RANDOM_STATE)ARTIFACT_DIR = f'{project_path}{course3_models}'feature_columns = joblib.load(f'{ARTIFACT_DIR}feature_columns.pkl')train_medians   = joblib.load(f'{ARTIFACT_DIR}train_medians.pkl')test_df = pd.read_csv(f'{project_path}{data_filepath}testing.csv')test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']# Raw features for the logistic PIPELINE (it encodes + scales internally)X_test_raw = test_df[numeric_features + categorical_features].copy()# Encoded matrix for the TREE models (manual dummies, aligned to training columns)test_enc = pd.get_dummies(test_df[numeric_features + categorical_features],                          columns=categorical_features, drop_first=True)test_enc = test_enc.reindex(columns=feature_columns, fill_value=0)test_enc = test_enc.fillna(train_medians)X_test, y_test = test_enc, test_df['DEPARTED']print(f"Testing: {X_test.shape[0]:,} | Encoded features: {X_test.shape[1]} | Raw cols: {X_test_raw.shape[1]}")

## Load the Tuned Models

In [ ]:
lr = joblib.load(f'{ARTIFACT_DIR}best_tuned_logistic_model.pkl')rf = joblib.load(f'{ARTIFACT_DIR}rf_tuned_f1.pkl')xgb = joblib.load(f'{ARTIFACT_DIR}xgb_tuned_f1.pkl')lr_pred = (lr.predict(X_test_raw) != 'E').astype(int)lr_prob = lr.predict_proba(X_test_raw)[:, 1]rf_pred = rf.predict(X_test)rf_prob = rf.predict_proba(X_test)[:, 1]xgb_pred = xgb.predict(X_test)xgb_prob = xgb.predict_proba(X_test)[:, 1]print("Models loaded and predictions generated.")

## Performance Comparison

In [ ]:
model_results = []models = {    'Regularized Logistic': {'preds': lr_pred, 'probs': lr_prob},    'Random Forest': {'preds': rf_pred, 'probs': rf_prob},    'XGBoost': {'preds': xgb_pred, 'probs': xgb_prob}}for name, d in models.items():    model_results.append({        'Model': name,        'Precision': precision_score(y_test, d['preds']),        'Recall': recall_score(y_test, d['preds']),        'F1 Score': f1_score(y_test, d['preds']),        'Avg Precision': average_precision_score(y_test, d['probs']),        'Brier Score': brier_score_loss(y_test, d['probs'])    })results_df = pd.DataFrame(model_results).set_index('Model')print(results_df.to_string(float_format="%.4f"))

## Precision-Recall Curves

In [ ]:
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']fig = go.Figure()for i, (name, prob) in enumerate([('Regularized Logistic', lr_prob),                                    ('Random Forest', rf_prob), ('XGBoost', xgb_prob)]):    prec, rec, _ = precision_recall_curve(y_test, prob)    ap = average_precision_score(y_test, prob)    fig.add_trace(go.Scatter(x=rec, y=prec, mode='lines',        name=f'{name} (AP={ap:.3f})', line=dict(color=colors[i], width=2)))baseline = y_test.mean()fig.add_trace(go.Scatter(x=[0, 1], y=[baseline, baseline], mode='lines',    name=f'No-skill ({baseline:.3f})', line=dict(color='gray', dash='dash')))fig.update_layout(height=450, title_text='Precision-Recall Curves')fig.update_xaxes(title_text='Recall')fig.update_yaxes(title_text='Precision')fig.show()

## Interpretability vs. Performance Trade-off

In [ ]:
import pandas as pdmax_f1 = results_df['F1 Score'].max()f1_lr = (results_df.loc['Regularized Logistic', 'F1 Score'] / max_f1) * 10f1_rf = (results_df.loc['Random Forest', 'F1 Score'] / max_f1) * 10f1_xgb = (results_df.loc['XGBoost', 'F1 Score'] / max_f1) * 10dimensions = ['F1 (scaled to 10)', 'Interpretability', 'Training Speed',              'Handles Non-linearity', 'Ease of Deployment']data = {    'Regularized Logistic': [f1_lr, 9, 9, 3, 9],    'Random Forest': [f1_rf, 5, 7, 8, 7],    'XGBoost': [f1_xgb, 3, 6, 9, 6]}df_radar = pd.DataFrame(data, index=dimensions)fig = go.Figure()for col in df_radar.columns:    fig.add_trace(go.Scatterpolar(        r=df_radar[col].values, theta=df_radar.index,        fill='toself', name=col, opacity=0.3, line_width=2    ))fig.update_layout(    polar=dict(radialaxis=dict(visible=True, range=[0, 10])),    showlegend=True,    title_text='Model Comparison: Performance vs. Practicality')fig.show()

## Recommendations| Your Priority | Recommended Model | Why ||:-------------|:-----------------|:----|| Explainability to advisors | Regularized Logistic Regression | Coefficients show clear factor contributions || Reliable risk scoring | Random Forest | Robust, handles messy data well || Maximum predictive accuracy | XGBoost | Typically highest AUC and F1 || Research publications | XGBoost | Best metrics for academic papers |**Two-model approach:** Regularized Logistic for stakeholder-facing outputs, Random Forest/XGBoost for backend risk scoring.**Next:** 4.2 Final Model Selection and Deployment